In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

Checking the Environment

In [0]:
print("RetailHub Data Platform")

print("Spark version:", spark.version)

spark.range(10).show()

Reading data

In [0]:
customers_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/workspace/default/retailhub/raw/customers.csv")
)

Displaying schema

In [0]:
customers_df.show(10)
customers_df.printSchema()

In [0]:
print("Rows:", customers_df.count())

Explicit Schema

In [0]:
from pyspark.sql.types import *

customer_schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("city", StringType(), True),
    StructField("age", IntegerType(), True)
])

In [0]:
customers_bronze_df = (
    spark.read
    .option("header", True)
    .schema(customer_schema)
    .csv(
        "/Volumes/workspace/default/retailhub/raw/customers.csv"
    )
)

In [0]:
customers_bronze_df.printSchema()

Adding Audit columns

In [0]:
customers_bronze_df = (
    customers_bronze_df.withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("load_date", current_date())
    .withColumn("source_file", lit("/Volumes/workspace/default/retailhub/raw/customers.csv"))
)

In [0]:
customers_broze_df.show(10, truncate= False)

Writing Broze data

In [0]:
(
    customers_bronze_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.bronze.customers"
    )
)

Reading data from Bronze Table

In [0]:
bronze_customers_df = spark.table("workspace.bronze.customers")
bronze_customers_df.show(10)

In [0]:
print("Bronze row count:",bronze_customers_df.count())

Reading data using SQL

In [0]:
%sql
SELECT 
    *
FROM    
    workspace.bronze.customers
LIMIT   
    10;

In [0]:
%sql
SELECT
    COUNT(*)
FROM    
    workspace.bronze.customers;

In [0]:
%sql
SELECT 
    *
FROM 
    workspace.bronze.customers
WHERE 
    email = 'invalid-email'
LIMIT 
    10;

In [0]:
%sql
SELECT
    customer_id,
    COUNT(*) AS record_count
FROM 
    workspace.bronze.customers
GROUP BY 
    customer_id
HAVING 
    COUNT(*) > 1
ORDER BY 
    record_count DESC;

Creating Schema for rest of the tables

In [0]:
product_schema = StructType([
    StructField("product_id", IntegerType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("brand", StringType(), True),
    StructField("cost_price", DoubleType(), True),
    StructField("selling_price", DoubleType(), True),
    StructField("stock", IntegerType(), True),
    StructField("product_status", StringType(), True)
])


store_schema = StructType([
    StructField("store_id", IntegerType(), True),
    StructField("store_name", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("region", StringType(), True),
    StructField("store_type", StringType(), True),
    StructField("opening_date", DateType(), True),
    StructField("store_status", StringType(), True)
])


order_schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("store_id", IntegerType(), True),
    StructField("order_date", DateType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("unit_price", DoubleType(), True),
    StructField("discount", DoubleType(), True),
    StructField("payment_method", StringType(), True),
    StructField("order_status", StringType(), True)
])


inventory_schema = StructType([
    StructField("inventory_id", IntegerType(), True),
    StructField("store_id", IntegerType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("stock_quantity", IntegerType(), True),
    StructField("reorder_level", IntegerType(), True),
    StructField("last_restock_date", DateType(), True),
    StructField("inventory_status", StringType(), True)
])

Reusable function for Ingestion

In [0]:
RAW_PATH = "/Volumes/workspace/default/retailhub/raw"


def ingest_to_bronze(
    file_name,
    schema,
    table_name
):
    
    source_path = f"{RAW_PATH}/{file_name}"
    
    
    df = (
        spark.read
        .option("header", True)
        .schema(schema)
        .csv(source_path)
    )
    

    bronze_df = (
        df.withColumn("ingestion_timestamp",current_timestamp())
        .withColumn("load_date",current_date())
        .withColumn("source_file",lit(source_path))
    )
    

    (
        bronze_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(
            f"workspace.bronze.{table_name}"
        )
    )
    

    return bronze_df

In [0]:
products_bronze_df = ingest_to_bronze(
    file_name="products.csv",
    schema=product_schema,
    table_name="products"
)


stores_bronze_df = ingest_to_bronze(
    file_name="stores.csv",
    schema=store_schema,
    table_name="stores"
)


orders_bronze_df = ingest_to_bronze(
    file_name="orders.csv",
    schema=order_schema,
    table_name="orders"
)


inventory_bronze_df = ingest_to_bronze(
    file_name="inventory.csv",
    schema=inventory_schema,
    table_name="inventory"
)

Verifying all bronze tables

In [0]:
print(
    "Customers:",
    spark.table("workspace.bronze.customers").count()
)

print(
    "Products:",
    spark.table("workspace.bronze.products").count()
)

print(
    "Stores:",
    spark.table("workspace.bronze.stores").count()
)

print(
    "Orders:",
    spark.table("workspace.bronze.orders").count()
)

print(
    "Inventory:",
    spark.table("workspace.bronze.inventory").count()
)